<a href="https://colab.research.google.com/github/AlejandroZapata-97/Analisis-armonicos/blob/main/Analisis_armonicos_corregido.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive

drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import h5py

#abrir archivo h5 en modo lectura
file = h5py.File('/content/drive/MyDrive/200721_202500_campaign_3.h5', 'r')

def print_structure(name, obj):
    print(name)
    print("Type:", type(obj))
    if isinstance(obj, h5py.Dataset):
        print("Shape:", obj.shape)
        print("Data Type:", obj.dtype)


#imprimir la estructura del archivo
file.visititems(print_structure)

FileNotFoundError: [Errno 2] Unable to synchronously open file (unable to open file: name = '/content/drive/MyDrive/200721_202500_campaign_3.h5', errno = 2, error message = 'No such file or directory', flags = 0, o_flags = 0)

In [ ]:
import numpy as np
from scipy import signal
import matplotlib.pyplot as plt
import h5py

# Cargar datos del archivo HDF5
file_path = '/content/drive/MyDrive/200721_202500_campaign_3.h5'
with h5py.File(file_path, 'r') as f:
    data = f['main_power_frp2_rogowski_480'][:10000000]  #Nombre archivo que se quiere analizar
    fs = f['/'].attrs['sample_clock_rate']  # Frecuencia de muestreo desde los atributos del archivo


# Graficar espectro de amplitud usando FFT(Transformada rapida de fourier)
def plot_fft_spectrum(data, fs, title):
    N = len(data)
    T = 1.0 / fs
    yf = np.fft.fft(data)
    xf = np.fft.fftfreq(N, T)[:N // 2]
    plt.plot(xf, 2.0 / N * np.abs(yf[:N // 2]))
    plt.grid()
    plt.title(title)
    plt.xlabel('Frecuencia [Hz]')
    plt.ylabel('Amplitud')
    plt.show()

# Graficar espectro de amplitud de la señal original
plot_fft_spectrum(data, fs, 'FFT de la Señal Original')

# Reducir la frecuencia de muestreo a 2000 Hz
new_fs = 2000  # Nueva frecuencia de muestreo
number_of_samples = int(len(data) * new_fs / fs)
downsampled_data = signal.resample(data, number_of_samples)

# Graficar espectro de amplitud de la señal submuestreada
plot_fft_spectrum(downsampled_data, new_fs, 'FFT de la Señal Submuestreada a 1000 Hz')

# Calcular la FFT de la señal submuestreada
N = len(downsampled_data)
T = 1.0 / new_fs
yf = np.fft.fft(downsampled_data)
xf = np.fft.fftfreq(N, T)[:N // 2]
yf_abs = 2.0 / N * np.abs(yf[:N // 2])

fundamental_frequency = 60

# Calcular el THD
def calculate_thd(yf_abs, xf, fundamental_freq):
    # Encontrar el índice de la frecuencia fundamental
    fundamental_index = np.argmax(xf >= fundamental_freq)
    fundamental_amplitude = yf_abs[fundamental_index]

    # Sumar las amplitudes de los armónicos
    harmonics_amplitude = 0
    for n in range(3, 16):
        harmonic_freq = fundamental_freq * n
        harmonic_index = np.argmin(np.abs(xf - harmonic_freq))
        harmonics_amplitude += yf_abs[harmonic_index]

    thd = harmonics_amplitude / fundamental_amplitude
    return thd * 100  # THD en porcentaje

# Calcular el THD
thd = calculate_thd(yf_abs, xf, fundamental_frequency)
print(f'THD de la señal submuestreada: {thd:.2f}%')


In [ ]:
import numpy as np
from scipy import signal
import matplotlib.pyplot as plt
import h5py

# Cargar datos del archivo HDF5
file_path = '/content/drive/MyDrive/200721_202500_campaign_3.h5'  #Archivo
with h5py.File(file_path, 'r') as f:
    data = f['main_power_frp2_rogowski_480'][:10000000]  #Nombre de el dataset que se quiere analizar
    fs = f['/'].attrs['sample_clock_rate']  # Frecuencia de muestreo desde los atributos del archivo

# Parámetros
fundamental_frequency = 60  # Frecuencia fundamental

# Filtrar armónicos del 3 al 15
filtered_data = data.copy()
for i in range(2, 16):
    f0 = i * fundamental_frequency
    if f0 < fs / 2:
        Q = 3.0  # Factor de calidad, ajusta según sea necesario
        w0 = f0 / (fs / 2)  # Normalizar la frecuencia
        b, a = signal.iirnotch(w0, Q)
        filtered_data = signal.filtfilt(b, a, filtered_data)

# Reducir la frecuencia de muestreo a 2000 Hz
new_fs = 2000  # Nueva frecuencia de muestreo
number_of_samples = int(len(filtered_data) * new_fs / fs)
downsampled_data = signal.resample(filtered_data, number_of_samples)

# Realizar la FFT y graficar el espectro de amplitud
def plot_fft_spectrum(data, fs, title):
    N = len(data)
    T = 1.0 / fs
    yf = np.fft.fft(data)
    xf = np.fft.fftfreq(N, T)[:N // 2]
    plt.plot(xf, 2.0 / N * np.abs(yf[:N // 2]))
    plt.grid()
    plt.title(title)
    plt.xlabel('Frecuencia [Hz]')
    plt.ylabel('Amplitud')
    plt.show()

plot_fft_spectrum(downsampled_data, new_fs, 'FFT de la Señal Filtrada y Submuestreada')

# Calcular la FFT de la señal submuestreada
N = len(downsampled_data)
T = 1.0 / new_fs
yf = np.fft.fft(downsampled_data)
xf = np.fft.fftfreq(N, T)[:N // 2]
yf_abs = 2.0 / N * np.abs(yf[:N // 2])

# Calcular el THD
def calculate_thd(yf_abs, xf, fundamental_freq):
    # Encontrar el índice de la frecuencia fundamental
    fundamental_index = np.argmax(xf >= fundamental_freq)
    fundamental_amplitude = yf_abs[fundamental_index]

    # Sumar las amplitudes de los armónicos
    harmonics_amplitude = 0
    for n in range(2, 16):
        harmonic_freq = fundamental_freq * n
        harmonic_index = np.argmin(np.abs(xf - harmonic_freq))
        harmonics_amplitude += yf_abs[harmonic_index]

    thd = harmonics_amplitude / fundamental_amplitude
    return thd * 100  # THD en porcentaje


# Calcular el THD
thd = calculate_thd(yf_abs, xf, fundamental_frequency)
print(f'THD de la señal submuestreada: {thd:.2f}%')
